# 03. Pandas 집계 분석 - 실습 문제

## 실습 안내
- 총 10개 문제
- 설비별, 제품별, 시간대별 집계 분석
- groupby, pivot_table, crosstab 활용
- 실제 제조 현장의 분석 시나리오

## 데이터 로드 및 전처리

In [1]:
import pandas as pd
import numpy as np

# 데이터 불러오기
production_df = pd.read_csv('../data/05_production.csv', encoding='utf-8-sig')
quality_df = pd.read_csv('../data/07_quality_inspection.csv', encoding='utf-8-sig', na_values=['\\N'])
equipment_df = pd.read_csv('../data/01_equipment.csv', encoding='utf-8-sig')
operation_df = pd.read_csv('../data/06_equipment_operation.csv', encoding='utf-8-sig')

# 기본 전처리
production_df['production_date'] = pd.to_datetime(production_df['production_date'])
production_df['defect_rate'] = (production_df['defect_quantity'] / production_df['actual_quantity'] * 100).round(2)
quality_df['inspection_time'] = pd.to_datetime(quality_df['inspection_time'])

print("데이터 로드 완료!")
print(f"생산: {len(production_df):,}건")
print(f"품질: {len(quality_df):,}건")
print(f"설비: {len(equipment_df):,}건")
print(f"설비운영: {len(operation_df):,}건")

데이터 로드 완료!
생산: 1,872건
품질: 37,417건
설비: 5건
설비운영: 3,304건


---
## 문제 1: 설비별 생산 통계

**시나리오**: 각 설비의 생산 성과를 종합적으로 분석하세요.

**요구사항**:
1. 설비별로 다음 지표 계산:
   - 생산 건수 (count)
   - 총 생산량 (actual_quantity의 sum)
   - 평균 생산량 (actual_quantity의 mean)
   - 총 불량수 (defect_quantity의 sum)
   - 평균 불량률 (defect_rate의 mean)
2. 총 생산량 기준 내림차순 정렬
3. 소수점 2자리로 반올림



In [2]:
production_df.columns

Index(['production_id', 'equipment_id', 'product_code', 'production_date',
       'start_time', 'end_time', 'target_quantity', 'actual_quantity',
       'good_quantity', 'defect_quantity', 'cycle_time', 'work_order_no',
       'lot_no', 'operator_id', 'shift', 'created_at', 'updated_at',
       'defect_rate'],
      dtype='object')

In [35]:
# 여기에 코드 작성
production_df.groupby('equipment_id').agg({'production_id' : 'count',
                                           'actual_quantity': ['sum','mean'],
                                           'defect_quantity' : 'sum',
                                           'defect_rate' : 'mean'})

production_id actual_quantity             defect_quantity  \
                     count             sum        mean             sum   
equipment_id                                                             
ASM-001                234           22485   96.089744            2726   
INJ-001                262           28163  107.492366            3017   
INJ-002                430           51958  120.832558            4504   
PRESS-001              468           52069  111.258547            5123   
PRESS-002              478           51929  108.638075            5506   

             defect_rate  
                    mean  
equipment_id              
ASM-001        12.116453  
INJ-001        10.759046  
INJ-002         8.701302  
PRESS-001       9.910855  
PRESS-002      10.675565

---
## 문제 2: 제품별 품질 분석

**시나리오**: 제품별 품질 수준을 파악하여 문제가 있는 제품을 찾으세요.

**요구사항**:
1. 제품별로 다음 지표 계산:
   - 생산 건수
   - 총 생산량
   - 평균 불량률
   - 최대 불량률
   - 최소 불량률
2. 평균 불량률이 높은 순서로 정렬



In [11]:
# 여기에 코드 작성
production_df.groupby('product_code').agg({'actual_quantity':['count', 'sum'],
                                           'defect_rate':['mean', 'max', 'min']})


actual_quantity        defect_rate            
                       count    sum        mean    max  min
product_code                                               
BUMPER-A                 648  71135   10.376590  19.48  2.0
DASH-C                   583  64741   10.121681  19.35  2.0
DOOR-B                   641  70728   10.158986  20.00  2.0

---
## 문제 3: 교대조별 생산 효율 비교

**시나리오**: 주간조와 야간조의 생산 효율을 비교 분석하세요.

**요구사항**:
1. 교대조(shift)별로 다음 지표 계산:
   - 생산 건수
   - 평균 생산량
   - 평균 불량률
   - 평균 사이클 타임
   - 목표 달성률 평균 (actual_quantity / target_quantity * 100)
2. 결과 해석: 어느 교대조가 더 효율적인가?



In [15]:
production_df['목표달성률'] = production_df['actual_quantity']/production_df['target_quantity'] * 100

In [16]:
# 여기에 코드 작성
production_df.groupby('shift').agg({'actual_quantity':['count', 'mean'],
                                    'cycle_time':'mean',
                                    'defect_rate':'mean',
                                    '목표달성률' : 'mean'
                                   })

actual_quantity             cycle_time defect_rate      목표달성률
                count        mean       mean        mean       mean
shift                                                              
DAY               936  109.711538   76.63562    8.815769  94.818320
NIGHT             936  111.019231   76.63563   11.629615  96.640882

---
## 문제 4: 설비 + 제품 복합 분석

**시나리오**: 각 설비가 어떤 제품을 얼마나 생산하는지 분석하세요.

**요구사항**:
1. 설비 + 제품별로 다음 집계:
   - 생산 건수
   - 총 생산량
   - 평균 불량률
2. 멀티인덱스 결과에서 특정 설비(예: INJ-001)의 제품별 데이터만 추출



In [20]:
# 여기에 코드 작성
pd.pivot_table(production_df, index='equipment_id', columns='product_code', values = 'actual_quantity', aggfunc=['sum', 'count'], fill_value=0)

sum                  count              
product_code BUMPER-A DASH-C DOOR-B BUMPER-A DASH-C DOOR-B
equipment_id                                              
ASM-001          6804   6779   8902       74     70     90
INJ-001         10858   9536   7769      104     85     73
INJ-002         18053  14766  19139      147    122    161
PRESS-001       18757  15670  17642      171    139    158
PRESS-002       16663  17990  17276      152    167    159

---
## 문제 5: 피벗 테이블 - 설비 x 제품 생산량 매트릭스

**시나리오**: 설비와 제품의 조합별 생산량을 한눈에 보는 표를 만드세요.

**요구사항**:
1. 피벗 테이블 생성:
   - 행: equipment_id
   - 열: product_code
   - 값: actual_quantity의 합계
   - 결측치는 0으로 채우기
2. 행/열 총계 추가 (margins=True)



In [21]:
# 여기에 코드 작성
pd.pivot_table(production_df, index='equipment_id', columns = 'product_code', values = 'actual_quantity', aggfunc='sum', margins = True, fill_value=0)

product_code,BUMPER-A,DASH-C,DOOR-B,All
equipment_id,,,,
ASM-001,6804,6779,8902,22485
INJ-001,10858,9536,7769,28163
INJ-002,18053,14766,19139,51958
PRESS-001,18757,15670,17642,52069
PRESS-002,16663,17990,17276,51929
All,71135,64741,70728,206604


---
## 문제 6: 피벗 테이블 - 설비 x 교대조 평균 불량률

**시나리오**: 설비별로 교대조에 따라 불량률이 어떻게 다른지 파악하세요.

**요구사항**:
1. 피벗 테이블 생성:
   - 행: equipment_id
   - 열: shift
   - 값: defect_rate의 평균
2. 소수점 2자리 반올림
3. 주간과 야간의 불량률 차이가 큰 설비 찾기



In [22]:
# 여기에 코드 작성
(pd.pivot_table(production_df, index = 'shift', columns = 'equipment_id', values = 'defect_rate', aggfunc = 'mean')).round(2)

equipment_id,ASM-001,INJ-001,INJ-002,PRESS-001,PRESS-002
shift,,,,,
DAY,10.67,9.36,7.2,8.53,9.34
NIGHT,13.56,12.16,10.2,11.29,12.01


---
## 문제 7: 불량 유형 분석 (crosstab)

**시나리오**: 제품별로 어떤 불량 유형이 많이 발생하는지 분석하세요.

**요구사항**:
1. 품질검사 데이터에서 불량(result='FAIL')만 필터링
2. 제품(product_code) x 불량코드(defect_code) 교차표 생성
3. 총계 포함
4. 각 제품의 주요 불량 유형 파악

**힌트**: 필터링 후 `pd.crosstab()`, margins=True

In [25]:
quality_df.head(2)

,inspection_id,production_id,equipment_id,product_code,inspection_time,inspection_type,result,defect_code,measurement_value,measurement_unit,inspector_id,lot_no,sample_size,notes,created_at
0,1,1,INJ-001,BUMPER-A,2024-01-01 09:41:18,FINAL,PASS,NaN,300.8279,mm,OP007,LOT2024010100101,1,NaN,2026-01-30 01:24:59
1,2,1,INJ-001,BUMPER-A,2024-01-01 08:17:24,FINAL,PASS,NaN,299.7696,mm,OP008,LOT2024010100101,1,NaN,2026-01-30 01:24:59


In [28]:
# 여기에 코드 작성
pd.crosstab(quality_df['product_code'], quality_df['defect_code'], margins = True)

defect_code,D001,D002,D003,D004,D005,D006,D007,D008,D009,D010,All
product_code,,,,,,,,,,,
BUMPER-A,1061,1458,720,385,1063,763,390,379,732,342,7293
DASH-C,656,963,984,339,1325,948,334,182,460,304,6495
DOOR-B,2829,1034,748,340,734,354,205,129,358,357,7088
All,4546,3455,2452,1064,3122,2065,929,690,1550,1003,20876


---
## 문제 8: 월별 생산 추이 분석

**시나리오**: 월별 생산량과 품질 추이를 분석하여 트렌드를 파악하세요.

**요구사항**:
1. production_date에서 년-월 추출 (dt.to_period('M'))
2. 월별로 다음 집계:
   - 생산 건수
   - 총 생산량
   - 평균 생산량
   - 평균 불량률
3. 시간 순서로 정렬
4. 처음 12개월 데이터 출력



In [ ]:
# 여기에 코드 작성
production_df['production_date'] = pd.to_datetime(production_df['production_date'])
pd.

---
## 문제 9: Transform을 이용한 설비별 표준화

**시나리오**: 각 설비의 생산량을 설비별 평균과 비교하여 성과를 평가하세요.

**요구사항**:
1. 설비별 평균 생산량을 계산하여 새 컬럼으로 추가 (transform)
2. 설비별 표준편차를 계산하여 새 컬럼으로 추가
3. 각 생산 건의 표준화 점수(Z-score) 계산:
   - Z-score = (실제값 - 평균) / 표준편차
4. Z-score가 2 이상인 생산 건 찾기 (매우 높은 생산량)



In [31]:
# 여기에 코드 작성
production_df['평균생산량']=production_df.groupby('equipment_id')['actual_quantity'].transform('mean')
production_df['평균대비편차'] = production_df['actual_quantity'] - production_df['평균생산량']
production_df['생산량표준편차'] = production_df.groupby('equipment_id')['actual_quantity'].transform('std')
production_df['z_score'] = production_df['평균대비편차'] / production_df['생산량표준편차']
production_df[production_df['z_score'] >= 2]

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,...,operator_id,shift,created_at,updated_at,defect_rate,목표달성률,평균생산량,평균대비편차,생산량표준편차,z_score


---
## 문제 10: 종합 설비 성능 대시보드

**시나리오**: 설비별 성능을 종합적으로 평가하는 대시보드 데이터를 만드세요.

**요구사항**:
1. 설비별로 다음 지표 모두 계산:
   - 생산 건수
   - 총 생산량
   - 평균 생산량
   - 생산량 표준편차
   - 평균 불량률
   - 평균 사이클 타임
   - 평균 목표 달성률
2. 성능 점수 계산 (사용자 정의):
   - 성능점수 = (평균생산량 / 평균사이클타임) * (100 - 평균불량률)
3. 성능 점수 기준 순위 매기기
4. 성능 점수 상위 5개 설비 출력



In [33]:
# 여기에 코드 작성
production_df.groupby('equipment_id').agg({'production_id' : 'count',
                                           'actual_quantity':[ 'sum', 'mean', 'std'],
                                           'cycle_time':'mean',
                                           'defect_rate':'mean',
                                           '목표달성률' : 'mean'
                                   })

production_id actual_quantity                        cycle_time  \
                     count             sum        mean        std       mean   
equipment_id                                                                   
ASM-001                234           22485   96.089744  16.575114  94.427137   
INJ-001                262           28163  107.492366  20.022352  71.714695   
INJ-002                430           51958  120.832558  22.112656  77.825558   
PRESS-001              468           52069  111.258547  20.685662  73.568333   
PRESS-002              478           51929  108.638075  20.195916  72.555900   

             defect_rate       목표달성률  
                    mean        mean  
equipment_id                          
ASM-001        12.116453   83.716011  
INJ-001        10.759046   93.584523  
INJ-002         8.701302  103.688416  
PRESS-001       9.910855   96.670964  
PRESS-002      10.675565   94.705213

---
## 수고하셨습니다!

### 학습 체크리스트
- [ ] groupby로 단일/다중 그룹화
- [ ] agg로 여러 집계 함수 동시 적용
- [ ] 명확한 컬럼명으로 집계 결과 생성
- [ ] pivot_table로 행/열 구조 변환
- [ ] crosstab으로 교차표 생성
- [ ] value_counts로 빈도 계산
- [ ] transform으로 그룹별 계산 결과 추가
- [ ] 복합 지표 계산 및 성능 평가

